# Municipal Tax Assessor Web Crawling
## Woonsocket, RI Property Data Analysis

This notebook demonstrates crawling a municipal tax assessor GIS website to extract property data and analyze property values by street and neighborhood.

In [ ]:
import logging
from collections import Counter

from WebCrawler.Serializers import Serializers
from WebCrawler.Spider import Spider

# Configure logging to track crawler activity
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s %(message)s",
)

In [ ]:
# Initialize and run the spider with progress tracking
# This crawler fetches street pages and their linked property pages
spider = Spider(
    start_url="https://gis.vgsi.com/WoonsocketRI/Streets.aspx",
    max_depth=2,  # Level 1: street pages, Level 2: individual property pages
    debug=False,
    show_progress=True,  # Enable real-time progress bar with visited/pending counts
    cache_dir=".assessor_cache",  # Cache responses to speed up repeat runs
)

# Run the crawl asynchronously
documents = await spider.run_async()
print(f"\n✓ Crawl complete: {len(documents)} pages fetched")

In [ ]:
# Display basic statistics about the crawl
print("=" * 60)
print("CRAWL STATISTICS")
print("=" * 60)
print(f"Total pages crawled: {len(documents)}")
print(f"Total links found: {sum(len(doc.links) for doc in documents)}")
print(f"  - Internal links: {sum(len(doc.internal_links) for doc in documents)}")
print(f"  - External links: {sum(len(doc.external_links) for doc in documents)}")
print()

# Show pages by HTTP status
status_counts = Counter(doc.status_code for doc in documents)
for status in sorted(status_counts.keys()):
    print(f"HTTP {status}: {status_counts[status]} pages")

In [ ]:
# Display page details with formatted output
print("\n" + "=" * 60)
print("PAGES CRAWLED")
print("=" * 60)
for i, doc in enumerate(documents, 1):
    title = doc.title if doc.title else "(no title)"
    internal_count = len(doc.internal_links)
    external_count = len(doc.external_links)
    print(f"\n[{i}] {title}")
    print(f"    URL: {doc.url}")
    print(f"    Status: HTTP {doc.status_code}")
    print(f"    Links: {internal_count} internal, {external_count} external")

In [ ]:
# Export to Pandas DataFrame for analysis
serializer = Serializers(documents)
df = serializer.to_pandas(include_html=False)

print(f"\nDataFrame shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print("\nFirst 10 rows:")
display(df.head(10))

In [ ]:
# Analyze property data: link distribution
print("\n" + "=" * 60)
print("LINK ANALYSIS")
print("=" * 60)

# Link type distribution
link_types = df["link_type"].value_counts()
print("\nLink type distribution:")
for link_type, count in link_types.items():
    print(f"  {link_type}: {count}")

# Pages by domain (if multiple domains crawled)
if len(df) > 0:
    unique_domains = df["domain"].nunique()
    print(f"\nUnique domains: {unique_domains}")
    if unique_domains > 1:
        print("\nPages by domain:")
        domain_counts = df["domain"].value_counts()
        for domain, count in domain_counts.items():
            print(f"  {domain}: {count} pages")

In [ ]:
# Export results to files for further analysis or archival
import os
from datetime import datetime

os.makedirs("data", exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Export to JSON (preserves nested link structure)
json_file = f"data/woonsocket_tax_assessor_{timestamp}.json"
serializer.to_json(json_file, include_html=False)
print(f"✓ Exported to JSON: {json_file}")

# Export to CSV (flattened format, one row per link)
csv_file = f"data/woonsocket_tax_assessor_{timestamp}.csv"
df.to_csv(csv_file, index=False)
print(f"✓ Exported to CSV: {csv_file}")

# Summary of exported data
print(f"\nExported {len(documents)} pages with {len(df)} link entries")